## Step 1: Load the Knowledge Base Documents

This cell reads all the `.md` files inside the `knowledge_base` folder and loads 
them into Python.

- `os.listdir(KB_DIR)` looks inside the folder and lists every file in it
- We only pick files ending in `.md` (our architecture documents)
- `TextLoader` opens each file and reads its text content
- Each loaded file becomes a `Document` object — this isn't just plain text, it also 
  remembers *which file it came from* (this is called metadata). We'll need this 
  later so the chatbot can tell you "this answer came from bauhaus_architecture.md" 
  when it responds

**Expected output:** "Loaded 8 documents" followed by a list of all 8 filenames. 
If you see fewer than 8, check that all your `.md` files are actually inside the 
`knowledge_base` folder and saved correctly.

In [2]:
import os
from langchain_community.document_loaders import TextLoader

KB_DIR = "knowledge_base"

docs = []
for filename in os.listdir(KB_DIR):
    if filename.endswith(".md"):
        path = os.path.join(KB_DIR, filename)
        loader = TextLoader(path, encoding="utf-8")
        docs.extend(loader.load())

print(f"Loaded {len(docs)} documents")
for d in docs:
    print("-", d.metadata["source"])

C:\Users\chama\AppData\Local\Temp\ipykernel_12232\2753012297.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\chama\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 8 documents
- knowledge_base\achaemenid_architecture.md
- knowledge_base\american_craftsman_style.md
- knowledge_base\american_foursquare_architecture.md
- knowledge_base\ancient_egyptian_architecture.md
- knowledge_base\art_deco_architecture.md
- knowledge_base\art_nouveau_architecture.md
- knowledge_base\baroque_architecture.md
- knowledge_base\bauhaus_architecture.md


## Step 2: Split Documents into Smaller Chunks

Right now each document is one big block of text (a few hundred words). This cell 
breaks each one into smaller pieces, roughly 500 characters each.

**Why we do this:** When someone asks a question, we don't want to hand the chatbot 
an entire document — we want to hand it just the *specific paragraph* that actually 
answers the question. Smaller chunks mean more precise, relevant search results later.

- `chunk_size=500` — each chunk is roughly 500 characters long
- `chunk_overlap=80` — each chunk shares 80 characters with the chunk before it, so 
  we don't accidentally cut a sentence or idea in half at a chunk boundary
- `separators` — tells it to prefer splitting at natural breaks first (a `##` heading, 
  then a blank line, then a new line, then a space) rather than cutting mid-word

**Expected output:** Since we have 8 documents and each is a few paragraphs long, 
you should see something like 8 documents → roughly 20-30 chunks (exact number depends 
on how long each doc is). Each chunk still remembers which original file it came from, 
shown in the "Metadata" line.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n## ", "\n\n", "\n", " "],
)

chunks = splitter.split_documents(docs)

print(f"Split {len(docs)} documents into {len(chunks)} chunks")
print("\nExample chunk:")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

Split 8 documents into 29 chunks

Example chunk:
# Achaemenid Architecture

## Overview
Achaemenid architecture developed under the Persian Achaemenid Empire (c. 550–330 BCE), 
best known through sites like Persepolis. It combined monumental scale with influences 
from Mesopotamian, Egyptian, and Anatolian traditions into a distinct imperial style.

Metadata: {'source': 'knowledge_base\\achaemenid_architecture.md'}


## Step 3: Turn Text into Searchable Numbers (Embeddings) and Store Them

Computers can't search text "by meaning" directly — they need numbers. This cell 
converts each chunk of text into a list of numbers (called a vector/embedding) that 
represents what that chunk is *about*.

- `HuggingFaceEmbeddings` loads a small, free AI model whose only job is turning text 
  into these number-vectors. This runs locally on your CPU — no internet needed after 
  the first download, and no Groq/API usage for this part
- Chunks that mean similar things end up with similar numbers. For example, a chunk 
  about "stucco cracking in Baroque buildings" and a chunk about "plaster damage in 
  ornate ceilings" would end up numerically close, even though they don't share many 
  exact words — this is what lets us search by *meaning* instead of exact keyword matching
- `Chroma.from_documents(...)` takes all our chunks, runs them through the embedding 
  model, and stores the results in a small local database
- `persist_directory="chroma_db"` saves this database to a folder on disk, so we don't 
  have to redo this embedding step every time we restart the notebook

**Expected output:** "Vector store created" and a chunk count matching what you saw 
in the last step (e.g. if you had 25 chunks before, it should say 25 here too).

**Note:** A new folder called `chroma_db` will appear in your project folder — that's 
the actual database file, don't delete it unless you want to rebuild everything from scratch.

In [4]:
import shutil
import os

if os.path.exists("chroma_db"):
    shutil.rmtree("chroma_db")
    print("Deleted old chroma_db folder")

Deleted old chroma_db folder


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db",
)

print("Vector store created and saved to chroma_db/")
print(f"Total chunks stored: {vectordb._collection.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4486.17it/s]


Vector store created and saved to chroma_db/
Total chunks stored: 29


In [9]:
from langchain_groq import ChatGroq

retriever = vectordb.as_retriever(search_kwargs={"k": 4})

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
)


test_results = retriever.invoke("What damage should I look for in Bauhaus buildings?")
print(f"Retrieved {len(test_results)} chunks:\n")
for r in test_results:
    print(f"[{r.metadata['source']}]")
    print(r.page_content[:150], "...\n")

Retrieved 4 chunks:

[knowledge_base\bauhaus_architecture.md]
## Conservation Considerations
- Flat roofs, common in this style, are more prone to water pooling and leaks than 
  pitched roofs if drainage isn't w ...

[knowledge_base\bauhaus_architecture.md]
- Because ornament was intentionally minimized, damage is often more visually obvious 
  and disruptive to the design intent than on more ornate histo ...

[knowledge_base\bauhaus_architecture.md]
# Bauhaus Architecture

## Overview
Bauhaus architecture emerged from the German Bauhaus school (1919–1933), rejecting 
ornament in favor of functiona ...

[knowledge_base\ancient_egyptian_architecture.md]
## Conservation Considerations
- Exposed relief carvings are vulnerable to sand erosion and salt crystallization, 
  especially near groundwater-affec ...



In [57]:
from langchain_core.prompts import PromptTemplate
QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are the NHPT Heritage Assistant, helping visitors understand historic sites.

Rules:
1. Answer ONLY using the CONTEXT below. If the context doesn't contain the answer, say 
   "I don't have that information in NHPT's records" rather than guessing.
2. Each context chunk is labeled with its source file. Only use chunks relevant to the 
   specific building style being asked about — ignore unrelated styles entirely.
3. Keep answers concise (2-4 sentences) and visitor-friendly.
4. After your answer, add a final line in this EXACT format listing only the source 
   files you actually used information from:
   SOURCES_USED: file1.md, file2.md

CONTEXT:
{context}

QUESTION: {question}

ANSWER:""",
)

STYLE_TO_FILE = {
    "achaemenid": "achaemenid_architecture.md",
    "foursquare": "american_foursquare_architecture.md",
    "craftsman": "american_craftsman_style.md",
    "egyptian": "ancient_egyptian_architecture.md",
    "art deco": "art_deco_architecture.md",
    "art nouveau": "art_nouveau_architecture.md",
    "baroque": "baroque_architecture.md",
    "bauhaus": "bauhaus_architecture.md",
}

def ask(question):
    matched_file = None
    for keyword, filename in STYLE_TO_FILE.items():
        if keyword in question.lower():
            matched_file = filename
            break

    if matched_file:
        source_path = f"knowledge_base\\{matched_file}"
        raw = vectordb.get(where={"source": source_path}, include=["documents", "metadatas"])
        results_text = raw["documents"]
        results_meta = raw["metadatas"]
    else:
        docs = retriever.invoke(question)
        results_text = [d.page_content for d in docs]
        results_meta = [d.metadata for d in docs]

    context = "\n\n".join(
        [f"[Source: {m['source']}]\n{t}" for t, m in zip(results_text, results_meta)]
    )
    sources = sorted(set(m["source"] for m in results_meta))

    prompt = QA_PROMPT.format(context=context, question=question)
    response = llm.invoke(prompt)

    print(f"Question: {question}\n")
    print(f"Answer: {response.content}\n")
    print(f"Sources: {', '.join(sources)}")

ask("What damage should I look for in Art Deco buildings?")


Question: What damage should I look for in Art Deco buildings?

Answer: Look for corrosion on decorative chrome and bronze details, cracks in terrazzo or cast stone along aggregate boundaries, and damage to pigmented Vitrolite glass panels. Also check cast relief panels for movement or falling hazards due to degraded mechanical fixings.  

SOURCES_USED: knowledge_base\art_deco_architecture.md

Sources: knowledge_base\art_deco_architecture.md


In [60]:
ask("How does that compare to what you just told me about Bauhaus?")

Question: How does that compare to what you just told me about Bauhaus?

Answer: I don't have that information in NHPT's records.  
SOURCES_USED:

Sources: knowledge_base\bauhaus_architecture.md


In [64]:
chat_history = []

STYLE_TO_FILE = {
    "achaemenid": "achaemenid_architecture.md",
    "foursquare": "american_foursquare_architecture.md",
    "craftsman": "american_craftsman_style.md",
    "egyptian": "ancient_egyptian_architecture.md",
    "art deco": "art_deco_architecture.md",
    "art nouveau": "art_nouveau_architecture.md",
    "baroque": "baroque_architecture.md",
    "bauhaus": "bauhaus_architecture.md",
}

def detect_styles(text):
    text_lower = text.lower()
    matched = []
    for keyword, filename in STYLE_TO_FILE.items():
        if keyword in text_lower and filename not in matched:
            matched.append(filename)
    return matched

def condense_question(question, history):
    if not history:
        return question
    history_text = "\n".join([f"Visitor: {q}\nAssistant: {a}" for q, a in history])
    prompt = f"""Given this conversation history and a follow-up question, rewrite the 
follow-up as a standalone question that includes any implied context from the history. 
Only output the rewritten question, nothing else.

HISTORY:
{history_text}

FOLLOW-UP QUESTION: {question}

STANDALONE QUESTION:"""
    return llm.invoke(prompt).content.strip()

def get_chunks_for_files(matched_files):
    all_text, all_meta = [], []
    for filename in matched_files:
        source_path = f"knowledge_base\\{filename}"
        raw = vectordb.get(where={"source": source_path}, include=["documents", "metadatas"])
        all_text.extend(raw["documents"])
        all_meta.extend(raw["metadatas"])
    return all_text, all_meta

def ask_with_memory(question, debug=True):
    search_question = condense_question(question, chat_history)
    if debug:
        print(f"[DEBUG] Standalone question: {search_question}")

    # Detect styles ONLY from the rewritten standalone question — not raw history
    matched_files = detect_styles(search_question)

    if matched_files:
        results_text, results_meta = get_chunks_for_files(matched_files)
        if debug:
            print(f"[DEBUG] Using full chunks from: {matched_files}")
    else:
        docs = retriever.invoke(search_question)
        results_text = [d.page_content for d in docs]
        results_meta = [d.metadata for d in docs]
        if debug:
            print("[DEBUG] No style detected, using open semantic search")

    context = "\n\n".join(
        [f"[Source: {m['source']}]\n{t}" for t, m in zip(results_text, results_meta)]
    )
    sources = sorted(set(m["source"] for m in results_meta))

    history_text = "\n".join([f"Visitor: {q}\nAssistant: {a}" for q, a in chat_history])

    prompt = f"""You are the NHPT Heritage Assistant, helping visitors understand historic sites.

Rules:
1. Answer ONLY using the CONTEXT below. If the context doesn't contain the answer, say 
   "I don't have that information in NHPT's records" rather than guessing.
2. Each context chunk is labeled with its source file. Only use chunks relevant to the 
   specific building style(s) being asked about.
3. Use the CONVERSATION HISTORY to understand follow-up questions.
4. Keep answers concise (2-4 sentences) and visitor-friendly.

CONVERSATION HISTORY:
{history_text if history_text else "(no previous messages)"}

CONTEXT:
{context}

VISITOR QUESTION: {question}

ANSWER:"""

    response = llm.invoke(prompt).content.strip()
    chat_history.append((question, response))

    print(f"\nVisitor: {question}\n")
    print(f"Assistant: {response}\n")
    print(f"Sources: {', '.join(sources)}")

In [65]:

ask_with_memory("What damage should I look for in Bauhaus buildings?")

[DEBUG] Standalone question: What damage should I look for in Bauhaus buildings?
[DEBUG] Using full chunks from: ['bauhaus_architecture.md']

Visitor: What damage should I look for in Bauhaus buildings?

Assistant: In Bauhaus buildings you’ll want to watch for a few key issues:

- **Flat roofs** can develop water pooling or leaks if drainage isn’t kept clear.  
- **Early reinforced concrete** may suffer from “concrete cancer” – corrosion of the steel reinforcement that cracks the concrete.  
- **Large glass curtain walls** need regular inspection for sealant wear, thermal gaps, and structural cracks.  

Because ornament is minimal, any visible damage stands out and can disrupt the building’s clean, functional look.

Sources: knowledge_base\bauhaus_architecture.md


In [ ]:
ask_with_memory("What damage should I look for in Bauhaus buildings?")

[DEBUG] Standalone question: What types of damage should I look for when inspecting Bauhaus buildings?
[DEBUG] Using full chunks from: ['bauhaus_architecture.md']

Visitor: What damage should I look for in Bauhaus buildings?

Assistant: In Bauhaus buildings you should check for:

- **Flat roofs** that may develop water pooling or leaks if drainage isn’t maintained.  
- **Early reinforced concrete** that can suffer “concrete cancer” – corrosion of steel reinforcement causing cracks.  
- **Large glass curtain walls** that need inspection for sealant wear, thermal gaps, and structural cracks.

Sources: knowledge_base\bauhaus_architecture.md


In [67]:
ask_with_memory("What can you tell me about Art Deco buildings instead?")

[DEBUG] Standalone question: What damage should I look for in Art Deco buildings?
[DEBUG] Using full chunks from: ['art_deco_architecture.md']

Visitor: What can you tell me about Art Deco buildings instead?

Assistant: Art Deco buildings are known for their geometric ornament—zigzags, chevrons, sunbursts, and stepped “ziggurat” forms—alongside vertical massing and stepped setbacks, especially in American skyscrapers. They often feature modern materials such as chrome, stainless steel, opaque colored glass (Vitrolite), and reinforced concrete, with motifs inspired by Egyptian, Aztec, and machine‑age imagery. When conserving these structures, watch for corrosion of decorative chrome and bronze, cracking of terrazzo or cast stone, damage to irreplaceable Vitrolite panels, and movement of cast relief panels that could pose a falling‑hazard risk.

Sources: knowledge_base\art_deco_architecture.md


In [ ]:
ask_with_memory("How does that compare to what you just told me about art decor?")

[DEBUG] Standalone question: How does the damage to look for in Bauhaus buildings compare to the damage to look for in Art Deco buildings?
[DEBUG] Using full chunks from: ['art_deco_architecture.md', 'bauhaus_architecture.md']

Visitor: How does that compare to what you just told me about art decor?

Assistant: Art Deco preservation centers on protecting its ornate details—chrome, bronze, terrazzo, Vitrolite glass, and cast‑relief panels—while Bauhaus conservation focuses on structural elements: flat roofs, early reinforced concrete, and large glass curtain walls. In other words, Art Deco’s main risks are corrosion and cracking of decorative finishes, whereas Bauhaus’s are water pooling, concrete “cancer,” and sealant failure on curtain walls. Both styles require attention to the materials that define their look, but the specific damage concerns differ because of their distinct design philosophies.

Sources: knowledge_base\art_deco_architecture.md, knowledge_base\bauhaus_architecture.m

In [69]:
chat_history

[('What damage should I look for in Bauhaus buildings?',
  'In Bauhaus buildings you’ll want to watch for a few key issues:\n\n- **Flat roofs** can develop water pooling or leaks if drainage isn’t kept clear.  \n- **Early reinforced concrete** may suffer from “concrete cancer” – corrosion of the steel reinforcement that cracks the concrete.  \n- **Large glass curtain walls** need regular inspection for sealant wear, thermal gaps, and structural cracks.  \n\nBecause ornament is minimal, any visible damage stands out and can disrupt the building’s clean, functional look.'),
 ('What damage should I look for in Bauhaus buildings?',
  'In Bauhaus buildings you should check for:\n\n- **Flat roofs** that may develop water pooling or leaks if drainage isn’t maintained.  \n- **Early reinforced concrete** that can suffer “concrete cancer” – corrosion of steel reinforcement causing cracks.  \n- **Large glass curtain walls** that need inspection for sealant wear, thermal gaps, and structural crack

In [ ]:
import tensorflow as tf
import numpy as np
import json
from tensorflow.keras.applications.resnet50 import preprocess_input

# Load your trained CV model (adjust path/filename to your actual saved file)
cv_model = tf.keras.models.load_model(
    "models/resnet_architectural_classifier_v2.keras",
    custom_objects={"preprocess_input": preprocess_input}
)

with open("class_indices_v1.json") as f:
    class_indices = json.load(f)

CONFIDENCE_THRESHOLD = 0.55  # below this, we treat the prediction as uncertain


def classify_image(image_path):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_batch = np.expand_dims(img_array, axis=0)

    preds = cv_model.predict(img_batch, verbose=0)[0]
    top_idx = np.argsort(preds)[::-1][:3]  # top 3 predictions

    top_predictions = [
        {"style": class_indices[str(i)], "confidence": float(preds[i])}
        for i in top_idx
    ]

    return {
        "image_path": image_path,
        "predicted_style": top_predictions[0]["style"],
        "confidence": top_predictions[0]["confidence"],
        "top_predictions": top_predictions,
    }


def cv_result_to_question(cv_result):
    style = cv_result["predicted_style"]
    confidence = cv_result["confidence"]

    if confidence < CONFIDENCE_THRESHOLD:
        alt = cv_result["top_predictions"][1]["style"]
        note = (
            f"[Note: our vision system is not fully confident — it estimates {confidence:.0%} "
            f"likelihood this is {style}, with {alt} as a possible alternative. "
            f"Please mention this uncertainty in your answer rather than stating the style as fact.] "
        )
    else:
        note = f"[Our vision system identified this with {confidence:.0%} confidence.] "

    return f"{note}A visitor photographed a building feature identified as {style}. What should they know about this style and what to look out for regarding its condition?"




FileNotFoundError: [Errno 2] No such file or directory: 'path/to/a/test_image.jpg'

In [72]:
# --- Test it end-to-end ---
cv_result = classify_image("2_800px-Quaker_Hill_Historic_District_-_140_Old_Norwich_Rd%2C_New_London_County_CT.jpg")
print("CV model output:", cv_result)

question = cv_result_to_question(cv_result)
print("\nGenerated question for chatbot:", question)

ask_with_memory(question)

CV model output: {'image_path': '2_800px-Quaker_Hill_Historic_District_-_140_Old_Norwich_Rd%2C_New_London_County_CT.jpg', 'predicted_style': 'American craftsman style', 'confidence': 0.9905807375907898, 'top_predictions': [{'style': 'American craftsman style', 'confidence': 0.9905807375907898}, {'style': 'American Foursquare architecture', 'confidence': 0.009418146684765816}, {'style': 'Bauhaus architecture', 'confidence': 6.253486617424642e-07}]}

Generated question for chatbot: [Our vision system identified this with 99% confidence.] A visitor photographed a building feature identified as American craftsman style. What should they know about this style and what to look out for regarding its condition?
[DEBUG] Standalone question: I photographed a building feature that our vision system identified as American Craftsman style. What should I know about this architectural style, and what specific damage or condition issues should I look for?
[DEBUG] Using full chunks from: ['american_cra

In [74]:
# --- Test it end-to-end ---
cv_result = classify_image("2.jpg")
print("CV model output:", cv_result)

question = cv_result_to_question(cv_result)
print("\nGenerated question for chatbot:", question)

ask_with_memory(question)

CV model output: {'image_path': '2.jpg', 'predicted_style': 'Achaemenid architecture', 'confidence': 0.9997081160545349, 'top_predictions': [{'style': 'Achaemenid architecture', 'confidence': 0.9997081160545349}, {'style': 'Baroque architecture', 'confidence': 0.00022958499903324991}, {'style': 'Ancient Egyptian architecture', 'confidence': 5.8905818150378764e-05}]}

Generated question for chatbot: [Our vision system identified this with 100% confidence.] A visitor photographed a building feature identified as Achaemenid architecture. What should they know about this style and what to look out for regarding its condition?
[DEBUG] Standalone question: A visitor has photographed a building feature identified as Achaemenid architecture. What key characteristics define this style, and what specific condition issues should they be aware of when inspecting it?
[DEBUG] Using full chunks from: ['achaemenid_architecture.md']

Visitor: [Our vision system identified this with 100% confidence.] A 

In [81]:
import os

data_dir = "architectural-styles-dataset"

candidate_folders = ["American Foursquare architecture", "American craftsman style"]

low_confidence_examples = []

for style_folder in candidate_folders:
    folder_path = os.path.join(data_dir, style_folder)
    files = os.listdir(folder_path)

    for fname in files[:]:  # check first 20 images in each folder
        test_image_path = os.path.join(folder_path, fname)
        try:
            result = classify_image(test_image_path)
            print(f"{style_folder}/{fname}: {result['predicted_style']} ({result['confidence']:.2%})")
            if result['confidence'] < CONFIDENCE_THRESHOLD:
                low_confidence_examples.append(test_image_path)
        except Exception as e:
            print(f"Skipped {fname}: {e}")

print(f"\nFound {len(low_confidence_examples)} low-confidence examples:")
for path in low_confidence_examples:
    print(" -", path)

American Foursquare architecture/000968.jpg: American Foursquare architecture (93.15%)
American Foursquare architecture/000970.jpg: American Foursquare architecture (90.15%)
American Foursquare architecture/000973.jpg: American Foursquare architecture (86.96%)
American Foursquare architecture/000975.jpg: American Foursquare architecture (94.83%)
American Foursquare architecture/000976.jpg: American Foursquare architecture (69.47%)
American Foursquare architecture/000977.jpg: American craftsman style (74.87%)
American Foursquare architecture/000978.jpg: American Foursquare architecture (88.54%)
American Foursquare architecture/000979.jpg: American Foursquare architecture (94.03%)
American Foursquare architecture/000980.jpg: American craftsman style (70.03%)
American Foursquare architecture/000981.jpg: American Foursquare architecture (92.35%)
American Foursquare architecture/000982.jpg: American craftsman style (68.82%)
American Foursquare architecture/000983.jpg: American Foursquare ar

In [82]:
test_image_path = low_confidence_examples[0]  # pick the first one found

cv_result = classify_image(test_image_path)
print("CV model output:", cv_result)

question = cv_result_to_question(cv_result)
print("\nGenerated question for chatbot:", question)

ask_with_memory(question)

CV model output: {'image_path': 'architectural-styles-dataset\\American Foursquare architecture\\001023.jpg', 'predicted_style': 'American craftsman style', 'confidence': 0.5460534691810608, 'top_predictions': [{'style': 'American craftsman style', 'confidence': 0.5460534691810608}, {'style': 'American Foursquare architecture', 'confidence': 0.399495929479599}, {'style': 'Bauhaus architecture', 'confidence': 0.034278981387615204}]}

Generated question for chatbot: [Note: our vision system is not fully confident — it estimates 55% likelihood this is American craftsman style, with American Foursquare architecture as a possible alternative. Please mention this uncertainty in your answer rather than stating the style as fact.] A visitor photographed a building feature identified as American craftsman style. What should they know about this style and what to look out for regarding its condition?
[DEBUG] Standalone question: A visitor photographed a building feature that the vision system id

In [84]:
import shutil
import os

os.makedirs("low_confidence_evidence", exist_ok=True)

for path in low_confidence_examples:
    dest = os.path.join("low_confidence_evidence", os.path.basename(path))
    shutil.copy(path, dest)

print(f"Copied {len(low_confidence_examples)} images to low_confidence_evidence/")

Copied 32 images to low_confidence_evidence/


In [85]:
chat_history 

[('What damage should I look for in Bauhaus buildings?',
  'In Bauhaus buildings you’ll want to watch for a few key issues:\n\n- **Flat roofs** can develop water pooling or leaks if drainage isn’t kept clear.  \n- **Early reinforced concrete** may suffer from “concrete cancer” – corrosion of the steel reinforcement that cracks the concrete.  \n- **Large glass curtain walls** need regular inspection for sealant wear, thermal gaps, and structural cracks.  \n\nBecause ornament is minimal, any visible damage stands out and can disrupt the building’s clean, functional look.'),
 ('What damage should I look for in Bauhaus buildings?',
  'In Bauhaus buildings you should check for:\n\n- **Flat roofs** that may develop water pooling or leaks if drainage isn’t maintained.  \n- **Early reinforced concrete** that can suffer “concrete cancer” – corrosion of steel reinforcement causing cracks.  \n- **Large glass curtain walls** that need inspection for sealant wear, thermal gaps, and structural crack

In [86]:
import json

example_conversations = [
    {"question": q, "answer": a} for q, a in chat_history
]

with open("example_conversations.json", "w", encoding="utf-8") as f:
    json.dump(example_conversations, f, indent=2)

print(f"Saved {len(example_conversations)} conversations")

Saved 9 conversations
